# 12. Scaled Stale-Knowledge Test: Vector RAG vs a Temporal Graph on HoH (Outdated Wikipedia Facts)

**Presentation map:** Slide 38 (keep provenance and time on every fact), Slide 49 (knowledge changes; the graph must
be maintained), Slides 52–55 (bi-temporal edges: a newer fact *invalidates* the older one instead of deleting it).

Notebook 4 made this point on 56 hand-written runbook questions. This notebook re-tests it on **HoH** ("How Outdated
information Harms RAG", ACL 2025; Hugging Face `russwest404/HoH-QAs`, Apache-2.0): 111,972 questions built from
monthly Wikipedia snapshots (June–December 2024). Every question has a **current answer** with its evidence sentence
and one or more **outdated answers** with the evidence sentence of an older snapshot — e.g. a population count that
was revised, a person's new club, a renamed species.

**Design**
- **Sample:** a fixed-seed sample of 500 questions — 300 with one outdated version and 200 with two or more (the
  harder case: several stale versions compete with one current one). Questions whose current and outdated answers
  are identical are excluded.
- **Document pool (≈6,800 passages):** every version (current and outdated) of the evidence of the sampled
  questions, the versions of other questions from the same Wikipedia articles, and the versions of 2,000 random
  other questions as distractors. Each passage is the article title plus the evidence sentence, and carries the date
  of its snapshot. *Simplification:* HoH's evidence sentences stand in for full article versions.

**Three systems, same answer model and prompt:**

| System | What the answerer sees |
|---|---|
| (a) **Vector RAG** | the 5 most similar passages, no dates — what most RAG systems do |
| (b) **Vector RAG + dates** (strong baseline) | the same 5 passages, each with its snapshot date, and the instruction to prefer the newest |
| (c) **Temporal graph** | the LLM extracts one fact (subject, predicate, object) per passage; facts about the same slot of the same article are linked; the newest version is *current*, older versions get `invalid_at` and stay in the graph but are not retrieved; the answerer sees the 5 best *current* facts with their date and source sentence |

**Scoring:** an LLM judge (same model, blind to the system) classifies each response as giving the **current**
answer, an **outdated** answer, or **neither**. We also measure, without any LLM, whether an outdated version of the
gold evidence was in the retrieved context.

**Model:** `gpt-6-luna`, no reasoning, via the OpenAI Batch API (`smash_batch.py`). Cached in
`data/benchmarks/hoh/cache/`; `SMASH_MODE=replay` (default) re-runs the saved results without API calls.

In [1]:
import json, os, re, random
from collections import Counter, defaultdict
from typing import Literal
import numpy as np
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson, paired_boot

MODE = mode_from_env()
TEST = MODE == "test"
BASE = "data/benchmarks/hoh"
llm = LLM(f"{BASE}/cache" + ("_test" if TEST else ""))

rows = [json.loads(l) for l in open(f"{BASE}/hoh_qas.jsonl")]
for i, r in enumerate(rows):
    r["i"] = i
    r["doc"] = r["document"]["title"]
    r["date"] = r["last_modified_time"][:10]
same = lambda r: any(o["answer"].strip().lower() == r["answer"].strip().lower() for o in r["outdated_infos"])
print(f"HoH-QAs: {len(rows):,} questions over {len({r['document']['id'] for r in rows}):,} Wikipedia articles")
print("Outdated versions per question:", dict(Counter(min(len(r['outdated_infos']), 3) for r in rows)), "(3 = three or more)")

rng = random.Random(7)
one = sorted(r["i"] for r in rows if len(r["outdated_infos"]) == 1 and not same(r))
many = sorted(r["i"] for r in rows if len(r["outdated_infos"]) >= 2 and not same(r))
sample = rng.sample(one, 300) + rng.sample(many, 200)
if TEST:
    sample = sample[:12] + sample[300:308]
S = [rows[i] for i in sample]
for r in S:
    r["group"] = "1 outdated version" if len(r["outdated_infos"]) == 1 else "2+ outdated versions"
print(f"Sample: {len(S)} questions |", dict(Counter(r['group'] for r in S)))
ex = next(r for r in S if r["group"].startswith("2"))
print(f"\nExample: [{ex['doc']}] {ex['question']}\n  current ({ex['date']}): {ex['answer']}")
for o in ex["outdated_infos"]:
    print(f"  outdated ({o['last_modified_time'][:10]}): {o['answer']}")

HoH-QAs: 111,972 questions over 90,571 Wikipedia articles
Outdated versions per question: {1: 106501, 2: 4595, 3: 876} (3 = three or more)
Sample: 500 questions | {'1 outdated version': 300, '2+ outdated versions': 200}

Example: [Dangal (2016 film)] What is the ranking of "Dangal" as the highest-grossing sports film worldwide?
  current (2024-12-01): 19th
  outdated (2024-06-01): 16th
  outdated (2024-07-01): 18th


## The document pool — every snapshot version is a separate passage

In [2]:
def versions(r):
    # All snapshot versions of one question's evidence: (date, sentence, is_current).
    return [(r["date"], r["evidence"], True)] + [(o["last_modified_time"][:10], o["evidence"], False)
                                                 for o in r["outdated_infos"]]

docs = {r["doc"] for r in S}
pool_q = set(sample) | {r["i"] for r in rows if r["doc"] in docs}
others = sorted(set(range(len(rows))) - pool_q)
pool_q |= set(rng.sample(others, 100 if TEST else 2000))
passages, seen = [], set()
for i in sorted(pool_q):
    r = rows[i]
    for date, sent, cur in versions(r):
        text = f"{r['doc']}: {sent}"
        if (text, date) in seen:
            continue
        seen.add((text, date))
        passages.append({"doc": r["doc"], "date": date, "text": text, "q": i})
pid = defaultdict(list)
for j, p in enumerate(passages):
    pid[p["text"]].append(j)
print(f"Pool: {len(passages):,} passages from {len(pool_q):,} questions "
      f"({len({p['doc'] for p in passages}):,} articles, snapshot dates {min(p['date'] for p in passages)} to "
      f"{max(p['date'] for p in passages)})")

Pool: 6,550 passages from 2,986 questions (2,479 articles, snapshot dates 2024-06-01 to 2024-12-01)


## (a) and (b): dense retrieval over the pool

In [3]:
K = 5
P = llm.embed("passages", [p["text"] for p in passages])
Q = llm.embed("questions", [f"{r['doc']}: {r['question']}" for r in S])
vec_ids = [list(np.argsort(-(P @ Q[n]))[:K]) for n in range(len(S))]

def is_current_ev(r, j):
    return passages[j]["text"] == f"{r['doc']}: {r['evidence']}" and passages[j]["date"] == r["date"]

def is_outdated_ev(r, j):
    return any(passages[j]["text"] == f"{r['doc']}: {o['evidence']}" and passages[j]["date"] == o["last_modified_time"][:10]
               for o in r["outdated_infos"])

cur_hit = np.mean([any(is_current_ev(r, j) for j in ids) for r, ids in zip(S, vec_ids)])
old_hit = np.mean([any(is_outdated_ev(r, j) for j in ids) for r, ids in zip(S, vec_ids)])
print(f"Vector top-{K}: current evidence retrieved for {cur_hit:.1%} of questions; "
      f"an outdated version also in the context for {old_hit:.1%}")

Vector top-5: current evidence retrieved for 97.2% of questions; an outdated version also in the context for 99.2%


## (c) Temporal graph: extract → link versions → invalidate, don't delete (Slides 53–54)

One structured request per passage extracts the fact it states. Facts of the same article whose subject+predicate
embeddings are ≥ 0.85 similar describe the same **slot** (e.g. *Otago — population aged under 15*). Within a slot
the newest snapshot is current; every older fact gets `invalid_at` = the date of the next newer version. Nothing is
deleted: the superseded facts stay in the graph with their dates and source passage.

In [4]:
class Fact(BaseModel):
    subject: str = Field(description="the entity the fact is about (use the article title when the sentence does not name it)")
    predicate: str = Field(description="the attribute or relation, without the value, e.g. 'number of people aged under 15'")
    object: str = Field(description="the value, exactly as written (numbers, names, dates)")

FACT_SYS = ("Extract the single main fact that the passage states, as subject, predicate and object. The passage "
            "starts with the title of the Wikipedia article it comes from.")
res = llm.run("facts", {f"p{j}": LLM.body([{"role": "system", "content": FACT_SYS},
                                           {"role": "user", "content": p["text"]}], max_tokens=200, schema=Fact)
                        for j, p in enumerate(passages)}, parse=Fact)
facts = [res[f"p{j}"]["parsed"] or {"subject": p["doc"], "predicate": p["text"], "object": ""}
         for j, p in enumerate(passages)]
print(f"Facts extracted: {len(facts):,} ({sum(res[f'p{j}']['parsed'] is None for j in range(len(passages)))} unparsable)")

SP = llm.embed("slots", [f"{passages[j]['doc']} | {f['subject']} | {f['predicate']}" for j, f in enumerate(facts)], cap=400)
by_doc = defaultdict(list)
for j, p in enumerate(passages):
    by_doc[p["doc"]].append(j)
slot = list(range(len(passages)))                    # union-find over passages of the same article
def find(a):
    while slot[a] != a:
        slot[a] = slot[slot[a]]; a = slot[a]
    return a
for doc, js in by_doc.items():
    if len(js) < 2:
        continue
    S_ = SP[js] @ SP[js].T
    for a in range(len(js)):
        for b in range(a + 1, len(js)):
            if S_[a, b] >= 0.85:
                slot[find(js[a])] = find(js[b])
slots = defaultdict(list)
for j in range(len(passages)):
    slots[find(j)].append(j)
invalid_at = [None] * len(passages)
for js in slots.values():
    dates = sorted({passages[j]["date"] for j in js})
    for j in js:
        newer = [d for d in dates if d > passages[j]["date"]]
        if newer:
            invalid_at[j] = newer[0]
n_inv = sum(x is not None for x in invalid_at)
print(f"Slots: {len(slots):,} ({sum(len(v) > 1 for v in slots.values()):,} with several versions); "
      f"{n_inv:,} facts superseded (kept with invalid_at), {len(passages) - n_inv:,} current")

F = llm.embed("facts", [f"{passages[j]['doc']}: {f['subject']} {f['predicate']} {f['object']}" for j, f in enumerate(facts)], cap=400)
current = np.array([x is None for x in invalid_at])
graph_ids = []
for n in range(len(S)):
    sims = F @ Q[n]
    sims[~current] = -9                              # superseded facts are not retrieved
    graph_ids.append(list(np.argsort(-sims)[:K]))

ex_slot = next((js for js in slots.values() if len({passages[j]["date"] for j in js}) >= 3), None)
if ex_slot:
    print("\nOne slot with its versions:")
    for j in sorted(ex_slot, key=lambda j: passages[j]["date"]):
        f = facts[j]
        status = f"invalid_at {invalid_at[j]}" if invalid_at[j] else "CURRENT"
        print(f"  [{passages[j]['date']} | {status:<21}] {f['subject']} — {f['predicate']} — {f['object']}")

Facts extracted: 6,550 (0 unparsable)
Slots: 3,126 (2,386 with several versions); 3,289 facts superseded (kept with invalid_at), 3,261 current



One slot with its versions:
  [2024-06-01 | invalid_at 2024-07-01] Slovenia national football team — number of players who have made at least one appearance — 236
  [2024-07-01 | invalid_at 2024-11-01] Slovenia national football team — number of players who have made at least one appearance — 237
  [2024-11-01 | invalid_at 2024-12-01] Slovenia national football team — number of players who have made at least one appearance — 237
  [2024-12-01 | CURRENT              ] Slovenia national football team — number of players who have made at least one appearance — 238


## Answers and the judge

In [5]:
ANSWER_SYS = ("Answer the question from the provided passages only. Give a short answer on the last line as "
              "'Answer: ...'. If the passages do not contain the answer, say 'Answer: unknown'.")
DATED_SYS = ANSWER_SYS + (" Each passage shows the date of the Wikipedia snapshot it comes from; when passages "
                          "disagree, the most recent snapshot is correct.")

def ctx_plain(ids):
    return "\n".join(f"- {passages[j]['text']}" for j in ids)

def ctx_dated(ids):
    return "\n".join(f"- [snapshot {passages[j]['date']}] {passages[j]['text']}" for j in ids)

def ctx_graph(ids):
    return "\n".join(f"- [current since {passages[j]['date']}] {facts[j]['subject']} — {facts[j]['predicate']} — "
                     f"{facts[j]['object']}  (source: {passages[j]['text']})" for j in ids)

SYSTEMS = {"vector": (ANSWER_SYS, ctx_plain, vec_ids), "vector+dates": (DATED_SYS, ctx_dated, vec_ids),
           "graph": (DATED_SYS, ctx_graph, graph_ids)}
reqs = {}
for n, r in enumerate(S):
    for m, (sys_, ctx, ids) in SYSTEMS.items():
        reqs[f"{m}|{r['i']}"] = LLM.body([{"role": "system", "content": sys_},
                                         {"role": "user", "content": f"Passages:\n{ctx(ids[n])}\n\nQuestion ({r['doc']}): {r['question']}"}],
                                        max_tokens=200)
ANS = llm.run("answers", reqs)

class Verdict(BaseModel):
    verdict: Literal["current", "outdated", "neither"]

def judge_prompt(r, response):
    olds = "; ".join(sorted({o["answer"] for o in r["outdated_infos"]}))
    return ("Classify a model response to a question whose correct answer changed over time.\n"
            f"Question: {r['question']}\nCurrent (correct) answer: {r['answer']}\nOutdated answer(s): {olds}\n"
            f"Model response: {response}\n\nReturn 'current' if the response gives the current answer, 'outdated' if it "
            "gives an outdated answer (and not the current one), otherwise 'neither'.")
JUD = llm.run("judge", {k: LLM.body([{"role": "user", "content": judge_prompt(rows[int(k.split('|')[1])], a["text"])}],
                                    max_tokens=20, schema=Verdict) for k, a in ANS.items()}, parse=Verdict)
verdict = {k: (v["parsed"] or {"verdict": "neither"})["verdict"] for k, v in JUD.items()}
print(f"{len(ANS):,} answers judged:", dict(Counter(verdict.values())))

1,500 answers judged: {'current': 1216, 'neither': 184, 'outdated': 100}


## Results with 95% confidence intervals

In [6]:
GROUPS = ["1 outdated version", "2+ outdated versions", "ALL"]
RES = {}
print(f"{'group':<22}{'n':>5} | " + " | ".join(f"{m:^31}" for m in SYSTEMS))
print(f"{'':<22}{'':>5} | " + " | ".join(f"{'current [95% CI]':>19}{'outdated':>11}" for _ in SYSTEMS))
for g in GROUPS:
    rs = [r for r in S if g == "ALL" or r["group"] == g]
    cells = []
    for m in SYSTEMS:
        k = sum(verdict[f"{m}|{r['i']}"] == "current" for r in rs)
        o = sum(verdict[f"{m}|{r['i']}"] == "outdated" for r in rs)
        lo, hi = wilson(k, len(rs))
        RES[(g, m)] = (k, o, len(rs), lo, hi)
        cells.append(f"{f'{k / len(rs):.0%} [{lo:.0%}-{hi:.0%}]':>19}{o / len(rs):>11.0%}")
    print(f"{g:<22}{len(rs):>5} | " + " | ".join(cells))

print("\nRetrieved context (no LLM involved):")
for m, ids in (("vector / vector+dates", vec_ids), ("graph", graph_ids)):
    cur = np.mean([any(is_current_ev(r, j) for j in x) for r, x in zip(S, ids)])
    old = np.mean([any(is_outdated_ev(r, j) for j in x) for r, x in zip(S, ids)])
    print(f"  {m:<22} current evidence in context {cur:5.1%} | outdated evidence in context {old:5.1%}")

acc = lambda m: [float(verdict[f"{m}|{r['i']}"] == "current") for r in S]
DIFF = {pair: paired_boot(acc(pair[0]), acc(pair[1])) for pair in
        [("vector", "graph"), ("vector+dates", "graph"), ("vector", "vector+dates")]}
print("\nPaired differences in accuracy (95% bootstrap CI):")
for (a, b), (dm, lo, hi) in DIFF.items():
    print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]")

group                     n |             vector              |          vector+dates           |              graph             
                            |    current [95% CI]   outdated |    current [95% CI]   outdated |    current [95% CI]   outdated
1 outdated version      300 |       70% [65%-75%]        14% |       95% [92%-97%]         0% |       96% [93%-97%]         1%
2+ outdated versions    200 |       52% [45%-59%]        20% |       82% [77%-87%]         4% |       82% [76%-86%]         2%
ALL                     500 |       63% [59%-67%]        17% |       90% [87%-93%]         2% |       90% [87%-92%]         1%

Retrieved context (no LLM involved):
  vector / vector+dates  current evidence in context 97.2% | outdated evidence in context 99.2%
  graph                  current evidence in context 95.0% | outdated evidence in context 12.8%

Paired differences in accuracy (95% bootstrap CI):
  graph - vector: +27.0% [+22.8%, +31.2%]
  graph - vector+dates: -0.2% [-2.4%, 

### Where the systems disagree

A question that plain vector RAG got wrong because a stale passage was in its context.

In [7]:
r = next((r for n, r in enumerate(S) if verdict[f"vector|{r['i']}"] == "outdated" and verdict[f"graph|{r['i']}"] == "current"), S[0])
n = S.index(r)
print(f"Q ({r['doc']}): {r['question']}\nCurrent answer: {r['answer']} | outdated: "
      f"{', '.join(o['answer'] for o in r['outdated_infos'])}\n")
for m in SYSTEMS:
    print(f"{m:<13} -> {verdict[f'{m}|{r['i']}']:<9} {ANS[f'{m}|{r['i']}']['text'].strip().splitlines()[-1][:120]}")
print("\nVector context (with dates shown here for the reader):")
print(ctx_dated(vec_ids[n]))

Q (Jeremy Gordon): What is Jeremy Gordon's nationality?
Current answer: Guyanese | outdated: Canadian

vector        -> outdated  Answer: Canadian
vector+dates  -> current   Answer: Guyanese-born international cricketer.
graph         -> current   Answer: Guyanese

Vector context (with dates shown here for the reader):
- [snapshot 2024-06-01] Jeremy Gordon: Jeremy Gordon (born 20 January 1987) is a Canadian international cricketer.
- [snapshot 2024-07-01] Jeremy Gordon: Jeremy Gordon (born 20 January 1987) is a Guyanese-born international cricketer.
- [snapshot 2024-10-01] Josh Littlejohn: Josh Littlejohn was born on 16 September 1986 in Edinburgh, United Kingdom.
- [snapshot 2024-08-01] Jefry Marté: Jefry Leonal Marté Paulino (born June 21, 1991) is a Dominican professional baseball first baseman, third baseman and left fielder who is a free agent.
- [snapshot 2024-06-01] Jefry Marté: Jefry Leonal Marté Paulino (born June 21, 1991) is a Dominican professional baseball first baseman, t

## Cost

In [8]:
LEDGER = llm.print_ledger()

stage                           calls     input tok   output tok      USD  via Batch
answers                         1,500       517,776       16,959     0.03       100%
facts                           6,550     1,268,153      217,576     0.12       100%
judge                           1,500       232,232       19,784     0.02       100%
(embeddings)                        0       537,356            0     0.01         0%
TOTAL                           9,550     2,555,517      254,319     0.18


In [9]:
print("=" * 92)
print(f"RESULTS SUMMARY — Notebook 12: outdated facts on HoH, {len(S)} questions, {len(passages):,} passages, gpt-6-luna")
print("=" * 92)
for g in GROUPS:
    print(f"{g:<22} " + " | ".join(f"{m} {RES[(g, m)][0] / RES[(g, m)][2]:.0%} correct, {RES[(g, m)][1] / RES[(g, m)][2]:.0%} outdated"
                                   for m in SYSTEMS))
for (a, b), (dm, lo, hi) in DIFF.items():
    print(f"  {b} - {a}: {dm:+.1%} [{lo:+.1%}, {hi:+.1%}]" + ("  (resolvable)" if lo > 0 or hi < 0 else "  (not resolvable)"))
print("\nLecture message (Slides 38, 49, 52-55): stale versions are near-duplicates of the current text, so dense"
      "\nretrieval brings them into the context. Compare the three rows: what plain RAG loses, how much of it simply"
      "\nshowing dates recovers, and what remains for a graph that marks superseded facts and keeps them out.")

RESULTS SUMMARY — Notebook 12: outdated facts on HoH, 500 questions, 6,550 passages, gpt-6-luna
1 outdated version     vector 70% correct, 14% outdated | vector+dates 95% correct, 0% outdated | graph 96% correct, 1% outdated
2+ outdated versions   vector 52% correct, 20% outdated | vector+dates 82% correct, 4% outdated | graph 82% correct, 2% outdated
ALL                    vector 63% correct, 17% outdated | vector+dates 90% correct, 2% outdated | graph 90% correct, 1% outdated
  graph - vector: +27.0% [+22.8%, +31.2%]  (resolvable)
  graph - vector+dates: -0.2% [-2.4%, +2.0%]  (not resolvable)
  vector+dates - vector: +27.2% [+23.2%, +31.4%]  (resolvable)

Lecture message (Slides 38, 49, 52-55): stale versions are near-duplicates of the current text, so dense
retrieval brings them into the context. Compare the three rows: what plain RAG loses, how much of it simply
showing dates recovers, and what remains for a graph that marks superseded facts and keeps them out.
